# Phase 1 — Conformité : vérifier le droit de crawler
**Durée estimée : 30 min**

---

## Objectif
Avant d'écrire une seule ligne de code de scraping, un chef de projet IA doit **prouver que la collecte est autorisée**.  
Dans ce notebook, vous allez analyser les fichiers `robots.txt` de 3 sites et produire un tableau de décision argumenté.

## Ce que vous rendez à la fin
- Un tableau comparatif (3 lignes × 4 colonnes)
- Une décision argumentée : quel site scraper ? pourquoi ?

---
> **Rappel juridique rapide**  
> Un fichier `robots.txt` n'a **pas de valeur légale contraignante** (c'est une convention technique), mais l'ignorer délibérément peut être retenu contre vous dans une procédure (CJUE, Ryanair/PR Aviation, 2015). Les CGU, elles, ont une valeur contractuelle.

In [1]:
# --- Installation (décommentez si vous êtes sur Google Colab) ---
# !pip install requests pandas

In [6]:
import requests
import pandas as pd
from urllib.robotparser import RobotFileParser
from IPython.display import display, Markdown

print("Imports OK ✓")

Imports OK ✓


---
## Étape 1 — Récupérer et afficher les robots.txt

Exécutez la cellule ci-dessous pour télécharger et afficher le contenu brut des 3 fichiers.  
**Lisez-les attentivement** avant de répondre aux questions.

In [7]:
SITES = {
    "data.gouv.fr": "https://www.data.gouv.fr/robots.txt",
    "lemonde.fr":   "https://www.lemonde.fr/robots.txt",
    "wikipedia.fr": "https://fr.wikipedia.org/robots.txt",
    "indeed.fr":    "https://fr.indeed.com/robots.txt",   # ← site d'offres d'emploi
}

robots_raw = {}

for nom, url in SITES.items():
    try:
        r = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        robots_raw[nom] = r.text
        print(f"✅ {nom} — {len(r.text)} caractères récupérés")
    except Exception as e:
        robots_raw[nom] = ""
        print(f"❌ {nom} — Erreur : {e}")

✅ data.gouv.fr — 2170 caractères récupérés
✅ lemonde.fr — 11995 caractères récupérés
✅ wikipedia.fr — 16761 caractères récupérés
✅ indeed.fr — 14534 caractères récupérés


In [8]:
# Affiche le robots.txt d'un site — changez le nom pour voir les autres
SITE_A_AFFICHER = "data.gouv.fr"  # <-- modifiez : "lemonde.fr", "wikipedia.fr" ou "indeed.fr"

print(f"=== robots.txt de {SITE_A_AFFICHER} ===")
print(robots_raw[SITE_A_AFFICHER][:3000])
print("[...]") if len(robots_raw[SITE_A_AFFICHER]) > 3000 else None

=== robots.txt de data.gouv.fr ===
User-agent: *
Disallow: /admin
Disallow: /en/admin
Disallow: /fr/admin
Disallow: /es/admin
Disallow: /datasets/search?*
Disallow: /en/datasets/search?*
Disallow: /fr/datasets/search?*
Disallow: /es/datasets/search?*
Disallow: /reuses/search?*
Disallow: /en/reuses/search?*
Disallow: /fr/reuses/search?*
Disallow: /es/reuses/search?*
Disallow: /dataservices/search?*
Disallow: /en/dataservices/search?*
Disallow: /fr/dataservices/search?*
Disallow: /es/dataservices/search?*
Disallow: /organizations?*
Disallow: /en/organizations?*
Disallow: /fr/organizations?*
Disallow: /es/organizations?*
Disallow: /datasets.csv
Disallow: /en/datasets.csv
Disallow: /fr/datasets.csv
Disallow: /es/datasets.csv
Disallow: /resources.csv
Disallow: /en/resources.csv
Disallow: /fr/resources.csv
Disallow: /es/resources.csv
Disallow: /reuses.csv
Disallow: /en/reuses.csv
Disallow: /fr/reuses.csv
Disallow: /es/reuses.csv
Disallow: /dataservices.csv
Disallow: /en/dataservices.csv
Disa

In [9]:
# Affiche le robots.txt d'un site — changez le nom pour voir les autres
SITE_A_AFFICHER = "indeed.fr"  # <-- modifiez : "lemonde.fr", "wikipedia.fr" ou "indeed.fr"

print(f"=== robots.txt de {SITE_A_AFFICHER} ===")
print(robots_raw[SITE_A_AFFICHER][:3000])
print("[...]") if len(robots_raw[SITE_A_AFFICHER]) > 3000 else None

=== robots.txt de indeed.fr ===
User-agent: *
Allow: /
Allow: /hire/*?*isid=
Allow: /personeel/*?*isid=
Allow: /reclutamiento/*?*isid=
Allow: /recruiting/*?*isid=
Allow: /recrutement/*?*isid=
Disallow: /*rt=nc
Disallow: /*&alid=
Disallow: /*&calert=
Disallow: /*&iafilter=
Disallow: /*&mna=
Disallow: /*?rss
Disallow: /addlLoc/
Disallow: /ads/
Disallow: /advanced_search
Disallow: /alert
Disallow: /api/fetch/mc-anon
Disallow: /api/getrecjobs
Disallow: /applystart
Disallow: /cmp/_/
Disallow: /cmp/_c/claim/*
Disallow: /cmp/_rpc/
Disallow: /cmp/addlink
Disallow: /cmp/addvideo
Disallow: /cmp/Login*
Disallow: /cmp/*/analytics
Disallow: /cmp/*/company-questions
Disallow: /cmp/*/people
Disallow: /cmp/*/write-review
Disallow: /community/
Disallow: /company/*
Disallow: /conversion/
Disallow: /cookiemigrator/
Disallow: /create-resume/lp/
Disallow: /cdn-cgi/
Disallow: /%E4%BB%95%E4%BA%8B?
Disallow: /%E5%B7%A5%E4%BD%9C/
Disallow: /%E5%B7%A5%E4%BD%9C/CN/
Disallow: /%E5%B7%A5%E4%BD%9C/title/
Disallow: 

---
## Étape 2 — Tester des URLs spécifiques

La bibliothèque `urllib.robotparser` intégrée à Python peut lire un robots.txt et répondre automatiquement : "cette URL est-elle autorisée pour User-agent: *  ?"

Nous allons tester 2 URLs par site :
- une **URL "safe"** (page de contenu standard)
- une **URL "risquée"** (page de recherche, paramètre `?q=`, etc.)

In [10]:
TESTS = {
    "data.gouv.fr": (
        "https://www.data.gouv.fr/robots.txt",
        [
            "https://www.data.gouv.fr/fr/datasets/",            # page de liste — safe
            "https://www.data.gouv.fr/fr/search/?q=education",  # recherche — risquée
        ]
    ),
    "lemonde.fr": (
        "https://www.lemonde.fr/robots.txt",
        [
            "https://www.lemonde.fr/economie/",                 # rubrique — safe
            "https://www.lemonde.fr/recherche/?keywords=ia",    # recherche — risquée
        ]
    ),
    "wikipedia.fr": (
        "https://fr.wikipedia.org/robots.txt",
        [
            "https://fr.wikipedia.org/wiki/Intelligence_artificielle",  # article — safe
            "https://fr.wikipedia.org/w/index.php?search=IA",          # recherche interne — risquée
        ]
    ),
    "indeed.fr": (
        "https://fr.indeed.com/robots.txt",
        [
            "https://fr.indeed.com/emplois?q=data+scientist&l=Paris",              # recherche — safe ?
            "https://fr.indeed.com/viewjob?jk=c44fa6bcc73bd5d3&from=shareddesktop_copy",  # fiche offre — risquée
        ]
    ),
}

def lire_robots(robots_url: str) -> RobotFileParser:
    """Charge un robots.txt en gérant le BOM UTF-8 (ex. Wikipedia)."""
    rp = RobotFileParser()
    rp.set_url(robots_url)
    try:
        r = requests.get(robots_url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        contenu_propre = r.text.lstrip("﻿")  # strip BOM si présent
        rp.parse(contenu_propre.splitlines())
    except Exception as e:
        print(f"  ⚠️  Impossible de lire {robots_url} : {e}")
    return rp

resultats = []

for site, (robots_url, urls) in TESTS.items():
    rp = lire_robots(robots_url)
    for url in urls:
        autorisee = rp.can_fetch("*", url)
        delay     = rp.crawl_delay("*")
        resultats.append({
            "Site":        site,
            "URL testée":  url,
            "Autorisée ?": "✅ Oui" if autorisee else "❌ Non",
            "Crawl-delay": str(delay) + "s" if delay else "Non défini",
        })

df_tests = pd.DataFrame(resultats)
display(df_tests)

# Détection BOM
print("\n📌 BOM UTF-8 détecté sur :")
for nom, url in [
    ("data.gouv.fr", "https://www.data.gouv.fr/robots.txt"),
    ("lemonde.fr",   "https://www.lemonde.fr/robots.txt"),
    ("wikipedia.fr", "https://fr.wikipedia.org/robots.txt"),
    ("indeed.fr",    "https://fr.indeed.com/robots.txt"),
]:
    try:
        r = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        bom = r.text.startswith("﻿")
        print(f"  {nom} : {'⚠️  OUI → parsing manuel nécessaire' if bom else 'Non'}")
    except:
        pass

# ⚠️  Vérification manuelle pour Indeed — le parser Python donne un faux positif
print()
print("⚠️  Vérification manuelle Indeed — 'Disallow: /viewjob?' dans le bloc User-agent: * ?")
viewjob_disallow = "Disallow: /viewjob?" in robots_raw.get("indeed.fr", "")
print(f"   Trouvé dans robots.txt : {'OUI → /viewjob? est explicitement interdit aux bots anonymes' if viewjob_disallow else 'Non'}")

,Site,URL testée,Autorisée ?,Crawl-delay
0,data.gouv.fr,https://www.data.gouv.fr/fr/datasets/,✅ Oui,Non défini
1,data.gouv.fr,https://www.data.gouv.fr/fr/search/?q=education,✅ Oui,Non défini
2,lemonde.fr,https://www.lemonde.fr/economie/,✅ Oui,Non défini
3,lemonde.fr,https://www.lemonde.fr/recherche/?keywords=ia,✅ Oui,Non défini
4,wikipedia.fr,https://fr.wikipedia.org/wiki/Intelligence_art...,✅ Oui,Non défini
5,wikipedia.fr,https://fr.wikipedia.org/w/index.php?search=IA,❌ Non,Non défini
6,indeed.fr,https://fr.indeed.com/emplois?q=data+scientist...,✅ Oui,Non défini
7,indeed.fr,https://fr.indeed.com/viewjob?jk=c44fa6bcc73bd...,✅ Oui,Non défini



📌 BOM UTF-8 détecté sur :
  data.gouv.fr : Non
  lemonde.fr : Non
  wikipedia.fr : ⚠️  OUI → parsing manuel nécessaire
  indeed.fr : Non

⚠️  Vérification manuelle Indeed — 'Disallow: /viewjob?' dans le bloc User-agent: * ?
   Trouvé dans robots.txt : OUI → /viewjob? est explicitement interdit aux bots anonymes


---
## Comment lire ce tableau ?

### La colonne "Autorisée ?"

`can_fetch("*", url)` simule la question : *"un robot anonyme a-t-il le droit de visiter cette URL ?"*  
La réponse vient des règles `Allow` / `Disallow` du bloc `User-agent: *` du robots.txt.

**Points de vigilance sur nos résultats :**

| Ligne | Résultat affiché | Ce qu'il faut retenir |
|-------|-----------------|----------------------|
| data.gouv.fr `/fr/search/?q=` | ✅ Oui | Interdit `/fr/datasets/search?*` mais **pas** `/fr/search/` — deux chemins distincts. Subtilité à signaler. |
| lemonde.fr `/recherche/` | ✅ Oui | robots.txt autorise, **mais** Le Monde utilise JavaScript + tokens anti-bot. Autorisé ≠ faisable. |
| wikipedia `/wiki/…` | ✅ Oui | Autorisé. Sans le fix BOM, affichait ❌ à tort (voir section BOM). |
| wikipedia `/w/index.php?search=` | ❌ Non | Correct — `Disallow: /w/` dans le bloc `User-agent: *`. |
| indeed `/emplois?q=…` | ✅ Oui | Autorisé par le parser. |
| indeed `/viewjob?jk=…` | ✅ Oui | **⚠️ FAUX POSITIF** — voir explication ci-dessous. |

**La règle d'or** : robots.txt vous dit ce que le site *tolère techniquement*, les CGU vous disent ce que vous avez *le droit* de faire, et les protections anti-bot vous disent ce qui est *réellement faisable*. Un outil peut rendre un résultat "propre" tout en étant silencieusement faux — toujours vérifier manuellement.

---

### Indeed `/viewjob?` : le faux positif — deux bugs Python qui s'accumulent

Le robots.txt d'Indeed (bloc `User-agent: *`) contient :
```
User-agent: *
Allow: /          ← règle 1 : autorise tout par défaut
...
Disallow: /viewjob?   ← règle 2 : interdit les fiches offres
```

**Bug 1 — Python strip le `?` des patterns Disallow**  
`Disallow: /viewjob?` est stocké en interne comme `/viewjob` (sans `?`).  
Le `?` marque le début de la query string dans urllib, Python le coupe du chemin.  
Conséquence : le test porte sur `/viewjob`, pas `/viewjob?`.

**Bug 2 — Python utilise "premier match gagne", pas "chemin le plus long gagne"**  
Python évalue les règles **dans l'ordre** et s'arrête au premier qui matche :
```
URL testée : /viewjob?jk=abc

Règle 1 : Allow:/    → /viewjob?jk=abc commence par /  → match !  → return True  ← STOP
Règle 2 : Disallow:/viewjob  → jamais atteinte
```
`Allow: /` étant listée **avant** `Disallow: /viewjob?`, elle l'emporte toujours.

**Ce que dit la RFC 9309 (standard moderne, utilisé par Google) :**  
Le chemin **le plus long** (le plus spécifique) gagne.  
`/viewjob` (8 chars) bat `/` (1 char) → `Disallow: /viewjob?` devrait gagner → résultat correct = **interdit**.

```
Python stdlib  (1994 spec) : premier match  → Allow:/  gagne   → ✅ Oui  ← FAUX
RFC 9309 / Google          : chemin le plus long → Disallow:/viewjob? gagne  → ❌ Non  ← CORRECT
```

> 💡 **Changer le user-agent ne corrigerait pas ce bug** — il est dans la logique de résolution Allow/Disallow au sein du bloc, pas dans le ciblage du bon bloc.

**La vérification manuelle reste la seule source de vérité :**
```
grep -n "viewjob" robots.txt
→ Disallow: /viewjob?   (ligne 173, bloc User-agent: *)
→ Allow: /viewjob?      (ligne 202, bloc Googlebot/Claude-User/...)
```

---

### Indeed : une asymétrie révélatrice

Indeed réserve un bloc spécial aux moteurs de recherche et aux IA :
```
User-agent: Googlebot
User-agent: Claude-User
User-agent: ChatGPT-User
...
Allow: /viewjob?    ← les fiches offres sont autorisées pour eux
```

- Google, les LLMs → peuvent indexer les offres (visibilité SEO pour Indeed ✓)  
- Votre scraper → ne peut pas (protection du business model ✓)  
- Les CGU d'Indeed **interdisent explicitement** le scraping → **Risque juridique : Élevé**

---

### Qu'est-ce que le BOM UTF-8 ?

**BOM** = *Byte Order Mark* = caractère invisible `U+FEFF` en tête de certains fichiers.

```
Fichier normal  :  User-agent: * ...
Fichier avec BOM:  ﻿User-agent: * ...   ← U+FEFF invisible avant le U
```

`urllib.robotparser.read()` ne le strip pas → parsing silencieusement raté → `can_fetch()` retourne `False` pour **toutes** les URLs (cas Wikipedia). Corrigé ici par `.lstrip("﻿")` avant parsing manuel.

---
## Side test — user-agent nommé vs bot anonyme (Indeed)

En production, on ne se contente pas de `urllib.robotparser` : on utilise un parser **RFC 9309** (le standard actuel) qui implémente correctement la règle "chemin le plus long gagne".

Installez `reppy` si besoin : `pip install reppy` (ou décommentez la ligne dans la cellule suivante).

In [11]:
# -----------------------------------------------------------------------
# Side test : même URL, 3 identités différentes
# stdlib Python (spec 1994) vs implémentation RFC 9309 (chemin le plus long)
# -----------------------------------------------------------------------

def can_fetch_rfc9309(robots_content: str, useragent: str, url: str) -> bool:
    """
    Implémentation RFC 9309 : le chemin le plus long (le plus spécifique) gagne.
    Contrairement à urllib.robotparser qui utilise 'premier match'.
    """
    from urllib.parse import urlparse
    path = urlparse(url).path + ("?" + urlparse(url).query if urlparse(url).query else "")

    # Collecte le bon bloc user-agent (exact match d'abord, sinon *)
    ua_lower  = useragent.lower().split("/")[0]
    blocs     = {"*": []}
    current   = None

    for line in robots_content.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if line.lower().startswith("user-agent:"):
            agent = line.split(":", 1)[1].strip().lower()
            current = agent
            if agent not in blocs:
                blocs[agent] = []
        elif line.lower().startswith(("allow:", "disallow:")) and current is not None:
            directive, rule_path = line.split(":", 1)
            blocs[current].append((directive.lower().strip(), rule_path.strip()))

    regles = blocs.get(ua_lower, blocs.get("*", []))

    # RFC 9309 : parmi toutes les règles qui matchent, la plus longue gagne
    meilleure_longueur = -1
    meilleure_decision = True  # par défaut : autorisé

    for directive, rule_path in regles:
        if path.startswith(rule_path) or rule_path == "":
            if len(rule_path) > meilleure_longueur:
                meilleure_longueur = len(rule_path)
                meilleure_decision = (directive == "allow")

    return meilleure_decision


# -----------------------------------------------------------------------
URL_OFFRE  = "https://fr.indeed.com/viewjob?jk=c44fa6bcc73bd5d3"
contenu    = robots_raw.get("indeed.fr", "")

agents_test = [
    ("*",               "Bot anonyme (votre scraper)"),
    ("Googlebot",       "Googlebot (Google Search)"),
    ("Claude-User",     "Claude-User (Anthropic)"),
    ("TP-Scraping/1.0", "User-agent custom (inconnu d'Indeed)"),
]

print("=" * 65)
print("  SIDE TEST — /viewjob? sur Indeed")
print("  Qui a le droit de crawler cette fiche offre ?")
print("=" * 65)

print(f"\n{'Agent':<22} {'stdlib Python (1994)':>20}   {'RFC 9309 (correct)':>18}")
print("─" * 65)

rp_std = lire_robots("https://fr.indeed.com/robots.txt")

for agent, label in agents_test:
    std  = rp_std.can_fetch(agent, URL_OFFRE)
    rfc  = can_fetch_rfc9309(contenu, agent, URL_OFFRE)
    bug  = " ← BUG" if std != rfc else ""
    print(f"  {label:<30}  {'✅ Oui' if std else '❌ Non':>6}          {'✅ Oui' if rfc else '❌ Non':>6}{bug}")

print("─" * 65)
print()
print("Lecture directe (vérité terrain) :")
print(f"  User-agent: *         Disallow: /viewjob?  → ❌ interdit aux bots anonymes")
print(f"  User-agent: Googlebot  Allow: /viewjob?    → ✅ autorisé aux moteurs indexeurs")
print()
print("Pourquoi stdlib est faux pour * :")
print("  Allow:/  apparaît AVANT Disallow:/viewjob? → premier match → Allow gagne")
print("  RFC 9309 : /viewjob (8 chars) > / (1 char) → chemin le plus long → Disallow gagne")

  SIDE TEST — /viewjob? sur Indeed
  Qui a le droit de crawler cette fiche offre ?

Agent                  stdlib Python (1994)   RFC 9309 (correct)
─────────────────────────────────────────────────────────────────
  Bot anonyme (votre scraper)      ✅ Oui           ❌ Non ← BUG
  Googlebot (Google Search)        ✅ Oui           ✅ Oui
  Claude-User (Anthropic)          ✅ Oui           ✅ Oui
  User-agent custom (inconnu d'Indeed)   ✅ Oui           ❌ Non ← BUG
─────────────────────────────────────────────────────────────────

Lecture directe (vérité terrain) :
  User-agent: *         Disallow: /viewjob?  → ❌ interdit aux bots anonymes
  User-agent: Googlebot  Allow: /viewjob?    → ✅ autorisé aux moteurs indexeurs

Pourquoi stdlib est faux pour * :
  Allow:/  apparaît AVANT Disallow:/viewjob? → premier match → Allow gagne
  RFC 9309 : /viewjob (8 chars) > / (1 char) → chemin le plus long → Disallow gagne


---
## Étape 3 — Tableau comparatif (à compléter)

Complétez le dictionnaire `votre_analyse` ci-dessous avec vos conclusions.  
**Soyez précis** : citez des règles réelles trouvées dans les robots.txt.

In [12]:
# ✏️  COMPLÉTEZ CE DICTIONNAIRE avec vos observations
votre_analyse = [
    {
        "Site":             "data.gouv.fr",
        "Pages interdites": "À compléter — ex: /fr/datasets/search?* est bloqué, mais pas /fr/search/",
        "Crawl-delay":      "À compléter — valeur trouvée ou pause appliquée",
        "Risque estimé":    "À compléter — Faible / Moyen / Élevé + 1 argument",
    },
    {
        "Site":             "lemonde.fr",
        "Pages interdites": "À compléter",
        "Crawl-delay":      "À compléter",
        "Risque estimé":    "À compléter",
    },
    {
        "Site":             "fr.wikipedia.org",
        "Pages interdites": "À compléter",
        "Crawl-delay":      "À compléter",
        "Risque estimé":    "À compléter",
    },
    {
        "Site":             "fr.indeed.com",
        "Pages interdites": "À compléter — indice : cherchez 'viewjob' dans le robots.txt",
        "Crawl-delay":      "À compléter",
        "Risque estimé":    "À compléter — indice : pensez aux CGU et à l'asymétrie User-agent: * vs Googlebot",
    },
]

df_analyse = pd.DataFrame(votre_analyse)
display(df_analyse)

,Site,Pages interdites,Crawl-delay,Risque estimé
0,data.gouv.fr,À compléter — ex: /fr/datasets/search?* est bl...,À compléter — valeur trouvée ou pause appliquée,À compléter — Faible / Moyen / Élevé + 1 argument
1,lemonde.fr,À compléter,À compléter,À compléter
2,fr.wikipedia.org,À compléter,À compléter,À compléter
3,fr.indeed.com,À compléter — indice : cherchez 'viewjob' dans...,À compléter,À compléter — indice : pensez aux CGU et à l'a...


---
## Étape 4 — Décision Chef de Projet

Répondez aux 3 questions ci-dessous dans la cellule Markdown suivante.

### Votre décision (à rédiger ici)

**1. Classement du moins risqué au plus risqué :**

| Rang | Site | Argument principal |
|------|------|--------------------|
| 1 (le moins risqué) | *à compléter* | *à compléter* |
| 2 | *à compléter* | *à compléter* |
| 3 (le plus risqué) | *à compléter* | *à compléter* |

**2. Site retenu pour la phase Collecte :**  
*à compléter — ex: Wikipedia, parce que...*

**3. Pause que vous appliquerez entre les requêtes :**  
*à compléter — ex: 2 secondes car le robots.txt ne précise pas de crawl-delay*

---
## ✅ Fin de la Phase 1

Avant de passer au notebook `02_collecte.ipynb`, vérifiez :
- [ ] Votre tableau comparatif est rempli avec des arguments tirés du robots.txt réel
- [ ] Vous avez testé au moins 2 URLs par site
- [ ] Vous avez choisi un site et justifié votre choix

**Passez maintenant à `02_collecte.ipynb` →**